In [1]:
import pandas as pd
import numpy as np

# Load cleaned Austria data (or use existing data with growth variables)
try:
    # Check if df already exists and has growth columns
    growth_cols = [col for col in df.columns if 'growth' in col or 'aagr' in col]
    if len(growth_cols) >= 4:  # Should have growth_2022, growth_2023, growth_2024, aagr_2024
        print(f"Data already loaded with {len(df.columns)} columns including growth variables")
        print("Available growth columns:", growth_cols)
    else:
        raise NameError("Data not fully loaded")
except NameError:
    # Load fresh data
    df = pd.read_pickle('../data/processed/austria_cleaned.pkl')
    print(f"Loaded fresh data: {df.shape}")
    print(f"Columns: {len(df.columns)}")
    print("Note: Growth variables need to be calculated")

Loaded fresh data: (46085, 27)
Columns: 27
Note: Growth variables need to be calculated


## Year-over-Year Growth Variables

Following the Belgium logic closely, with explicit adherence to the recommended rules for handling "n.a." values:

**Recommended Rules Applied:**
- **Rule 1**: If an input employee value needed for a calculation is unavailable, the output variable should also be unavailable.
- **Rule 2**: Do not convert unavailable values into zero growth.

**Growth Calculations:**
- `growth_2024 = (emp_2024 - emp_2023) / emp_2023`
- `growth_2023 = (emp_2023 - emp_2022) / emp_2022`
- `growth_2022 = (emp_2022 - emp_2021) / emp_2021`

**Rule Application**: If either required employee value is unavailable, set growth to "n.a."

This preserves the missingness logic from the raw data.

In [2]:
# Calculate year-over-year growth variables
# Following Rule 1: If input unavailable, output unavailable
# Following Rule 2: Do not convert unavailable values into zero growth
# Using numeric columns for calculation, then convert back to "n.a." for missing values

# Growth 2024: (emp_2024 - emp_2023) / emp_2023
df['growth_2024'] = (df['emp_2024_num'] - df['emp_2023_num']) / df['emp_2023_num']

# Growth 2023: (emp_2023 - emp_2022) / emp_2022
df['growth_2023'] = (df['emp_2023_num'] - df['emp_2022_num']) / df['emp_2022_num']

# Growth 2022: (emp_2022 - emp_2021) / emp_2021
df['growth_2022'] = (df['emp_2022_num'] - df['emp_2021_num']) / df['emp_2021_num']

# Convert NaN results back to "n.a." strings to match the missingness logic
for col in ['growth_2024', 'growth_2023', 'growth_2022']:
    df[col] = df[col].fillna('n.a.')

print("Growth variables created:")
for col in ['growth_2024', 'growth_2023', 'growth_2022']:
    na_count = (df[col] == 'n.a.').sum()
    valid_count = (df[col] != 'n.a.').sum()
    print(f"  {col}: {valid_count} valid values, {na_count} 'n.a.' values")

Growth variables created:
  growth_2024: 25850 valid values, 20235 'n.a.' values
  growth_2023: 30820 valid values, 15265 'n.a.' values
  growth_2022: 31979 valid values, 14106 'n.a.' values


In [3]:
# Inspect growth variables
print("Sample of growth values:")
for col in ['growth_2024', 'growth_2023', 'growth_2022']:
    print(f"\n{col} sample:")
    sample_values = df[col][df[col] != 'n.a.'].head(5)
    print(sample_values.values)
    print(f"  Min: {df[col][df[col] != 'n.a.'].min():.4f}")
    print(f"  Max: {df[col][df[col] != 'n.a.'].max():.4f}")
    print(f"  Mean: {df[col][df[col] != 'n.a.'].mean():.4f}")

Sample of growth values:

growth_2024 sample:
[0.14398795648795648 0.12037037037037036 0.013456751711263223
 -0.025634944164443126 0.12178327595261715]
  Min: -0.9971
  Max: 672.0000
  Mean: 0.2782

growth_2023 sample:
[-0.07692307692307693 0.0 0.046053702196908054 0.019382265831201017
 0.15052030882846593]
  Min: -0.9985
  Max: 833.0000
  Mean: 0.3769

growth_2022 sample:
[-0.018518518518518517 -0.00561647499331372 -0.2702702702702703
 0.0018205037632801674 0.02744943796965243]
  Min: -0.9995
  Max: 549.0000
  Mean: 0.3978


## Average Annual Growth Rate (AAGR)

Following the Belgium logic exactly, with explicit adherence to the recommended rules for handling "n.a." values:

**Recommended Rules Applied:**
- **Rule 1**: If an input employee value needed for a calculation is unavailable, the output variable should also be unavailable.
- **Rule 4**: For growth-based classification, exclude firms with fewer than 10 employees in the relevant base year.

**Formula for aagr_2024:**
```
aagr_2024 = (emp_2024 / emp_2021)^(1/3) - 1
```
Then multiplied by 100 to get percentage.

**Rule Application (matching Belgium notebook):**
Set `aagr_2024 = "n.a."` if:
- `emp_2024` is unavailable (Rule 1)
- `emp_2021` is unavailable (Rule 1)
- `emp_2021 < 10` (Rule 4)

This excludes firms below the size threshold in the base year (2021).

In [4]:
# Calculate Average Annual Growth Rate (AAGR) for 2024
# Following Rule 1: If input unavailable, output unavailable
# Following Rule 4: Exclude firms with < 10 employees in base year
# Formula: (emp_2024 / emp_2021)^(1/3) - 1, then * 100

# First calculate the ratio
ratio = df['emp_2024_num'] / df['emp_2021_num']

# Calculate AAGR: (ratio)^(1/3) - 1, then * 100
aagr_values = (ratio ** (1/3) - 1) * 100

# Create the column as object type to hold both floats and strings
df['aagr_2024'] = aagr_values.astype(object)

# Apply the Belgium rule: set to "n.a." if:
# - emp_2024 is missing (Rule 1)
# - emp_2021 is missing (Rule 1)
# - emp_2021 < 10 (Rule 4)
mask_na = (
    df['emp_2024_num'].isna() |
    df['emp_2021_num'].isna() |
    (df['emp_2021_num'] < 10)
)
df.loc[mask_na, 'aagr_2024'] = 'n.a.'

print("AAGR 2024 calculated:")
na_count = (df['aagr_2024'] == 'n.a.').sum()
valid_count = (df['aagr_2024'] != 'n.a.').sum()
print(f"  Valid values: {valid_count}")
print(f"  'n.a.' values: {na_count}")
print(f"  Total: {valid_count + na_count}")

# Show sample of valid AAGR values
if valid_count > 0:
    sample_aagr = df['aagr_2024'][df['aagr_2024'] != 'n.a.'].head(5)
    print(f"\nSample AAGR values: {sample_aagr.values}")
    print(".2f")
    print(".2f")

AAGR 2024 calculated:
  Valid values: 22500
  'n.a.' values: 23585
  Total: 46085

Sample AAGR values: [1.6415054213749558 -6.4936255475965705 2.0272949552891273
 0.6791973065142232 2.4093808440041498]
.2f
.2f


In [5]:
# Save dataset with growth variables and AAGR
output_path = '../data/processed/austria_with_growth.pkl'
df.to_pickle(output_path)
print(f"Dataset with growth variables and AAGR saved to: {output_path}")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print("New columns added: growth_2024, growth_2023, growth_2022, aagr_2024")

Dataset with growth variables and AAGR saved to: ../data/processed/austria_with_growth.pkl
Shape: (46085, 31)
Columns: 31
New columns added: growth_2024, growth_2023, growth_2022, aagr_2024


## Summary

### Growth Variables Created
- `growth_2024`: Year-over-year growth from 2023 to 2024
- `growth_2023`: Year-over-year growth from 2022 to 2023
- `growth_2022`: Year-over-year growth from 2021 to 2022

### AAGR Variable Created
- `aagr_2024`: Average Annual Growth Rate from 2021 to 2024
- Formula: `(emp_2024 / emp_2021)^(1/3) - 1` × 100
- Excludes firms with `emp_2021 < 10` (size threshold)

### Calculation Method
Following Belgium logic exactly, with explicit adherence to the recommended rules for handling "n.a." values:

**Recommended Rules Applied:**
- **Rule 1**: If an input employee value needed for a calculation is unavailable, the output variable should also be unavailable.
- **Rule 2**: Do not convert unavailable values into zero growth.

- **Growth**: `growth_2024 = (emp_2024 - emp_2023) / emp_2023`
- **AAGR**: `(emp_2024 / emp_2021)^(1/3) - 1` × 100
- If either required employee value is missing → result = "n.a."

### Missingness Preservation
- Maintains the same missing data logic as raw employee data
- No imputation or estimation of missing values
- "n.a." strings used for unavailable calculations
- AAGR additionally excludes firms with < 10 employees in 2021

### Data Statistics
- **growth_2024**: 25,850 valid values, 20,235 "n.a."
- **growth_2023**: 30,820 valid values, 15,265 "n.a."
- **growth_2022**: 31,979 valid values, 14,106 "n.a."
- **aagr_2024**: 22,500 valid values, 23,585 "n.a."

### Classification Sample
For Austria classification (matching Belgium approach):
- Firms with valid 2021 and 2024 employee values
- At least 10 employees in 2021
- **Result**: 22,500 firms available for classification

### Output
- Dataset saved as `../data/processed/austria_with_growth.pkl`
- Shape: 46,085 rows × 31 columns
- Ready for classification and analysis steps